<a href="https://colab.research.google.com/github/AyalaCohen-1/DevoirCompilation/blob/main/Projet%20maths%20info.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#Projet Maths Info- Aviva Tordjman et Ayala Cohen

In [ ]:
#on importe les bibliotheques necessaires pour le projet
import pandas as pd
import numpy as np

####Partie 3: Construction de l'arbre

In [ ]:
#on commence par creer la structure des arbres
class Node:
    def __init__(
        self,
        feature=None,
        threshold=None,
        left=None,
        right=None,
        value=None
    ):
        self.feature = feature
        self.threshold = threshold
        self.left = left
        self.right = right
        self.value = value

In [ ]:
#partie a
#je choisis le critere de division Gini Impurity car il me semble plus simple que l'entropie

def gini(y):
    if len(y) == 0:
        return 0 #pour pas avoir de probleme de division par 0

    classes, counts = np.unique(y, return_counts=True)
    impurity = 1
    for count in counts:
        p = count / len(y)
        impurity -= p ** 2
    return impurity

In [ ]:
#partie b
#on utilise l'algorithme CART car c'est celui associé a la Gini impurity
#cet algorithme va tester les differentes valeurs de nos données pour trouver celles ou la coupure entre sain et malade est la plus nette (qu'on aura d'un cote clairement un plus grand pourcentage de malades que de l'autre coté)
#il va ainsi construire un arbre de decision de maniere recursive en testant les differentes valeurs et choisissant les meilleures

class DecisionTree: #on cree une nouvelle classe pour l'arbre de decision c'est ce que l'algorithme va créer
#on implemente ici l'algorithme de cart pour que l'arbre soit créé selon celui ci
    def __init__(self, max_depth=10):
        self.max_depth = max_depth
        self.root = None

    def fit(self, X, y):
        #on initialise la racine et lance la construction récursive de l'arbre
        self.root = self._build_tree(X, y, depth=0)

    def _build_tree(self, X, y, depth):
        # partie c
        #criteres d'arret mis ici pour le bon fonctionnement de la creation de notre arbre recursif
        # Si on atteint la profondeur max ou si le noeud n'a qu'1 seule classe on arrete de chercher, ca nous empeche d'avoir un algo infini
        if depth >= self.max_depth or len(np.unique(y)) <= 1:
            classes, counts = np.unique(y, return_counts=True)
            return Node(value=classes[np.argmax(counts)]) # renvoie la classe majoritaire

        #
        best_feat, best_thresh, best_gini = None, None, float("inf")

        # On teste toutes les colonnes et tous les seuils possibles
        for feat in range(X.shape[1]):
            for thresh in np.unique(X[:, feat]):
                # On sépare les indices selon la condition
                left_idx = np.where(X[:, feat] <= thresh)[0]
                right_idx = np.where(X[:, feat] > thresh)[0]

                # Si la division ne met pas tout d'un seul cote
                if len(left_idx) > 0 and len(right_idx) > 0:
                    # Calcul du Gini pondéré des deux côtés
                    gini_val = (len(left_idx) * gini(y[left_idx]) + len(right_idx) * gini(y[right_idx])) / len(y)

                    # On sauvegarde si c'est le meilleur score
                    if gini_val < best_gini:
                        best_gini, best_feat, best_thresh = gini_val, feat, thresh

        # si aucune division n'ameliore le modele, on cree une feuille
        if best_feat is None:
            classes, counts = np.unique(y, return_counts=True)
            return Node(value=classes[np.argmax(counts)])

        left_idx = np.where(X[:, best_feat] <= best_thresh)[0]
        right_idx = np.where(X[:, best_feat] > best_thresh)[0]

        # le noeud s'appelle lui-même pour créer les branches suivantes et recommencer le processus de plus haut, c'est ca appliqué a chaque noeud créé de l'arbre qui va finir par créer l'arbre en entier
        return Node(
            feature=best_feat,
            threshold=best_thresh,
            left=self._build_tree(X[left_idx], y[left_idx], depth + 1),
            right=self._build_tree(X[right_idx], y[right_idx], depth + 1)
        )

    def predict(self, X):
        #Prédit la classe pour de nouvelles données
        # fonction interne pour descendre une ligne de données dans l'arbre
        def _traverse(x, node):
            if node.value is not None:
                return node.value # On a atteint une feuille, on donne le diagnostic associé a cette feuille (ya que 2 diagnostics possibles sain ou malade)

            # On descend à gauche ou à droite selon la condition
            if x[node.feature] <= node.threshold:
                return _traverse(x, node.left)
            return _traverse(x, node.right)

        # Applique la traversée à chaque ligne de X
        return np.array([_traverse(x, self.root) for x in X])